In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.precision", 16)

PROJECT_ROOT = Path(r"D:\TU_DataScience\22_Data Mining Cup\TS-Forecasting-Project")
TEST_FILE = PROJECT_ROOT / "data" / "raw" / "ballengewichte_rueckhalt.csv"

df = pd.read_csv(TEST_FILE)
df.columns = [str(c).strip() for c in df.columns]

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Shape: (345, 15)
Columns: ['Datum', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14']


,Datum,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,2025-01-01,0.2698455028712642,0.1270289971020676,0.1182996426476078,0.0616921791384429,0.1222465198143901,0.0430956317670275,0.0000000000000000,0.0464380322506089,0.0602521023343467,0.1511013920742439,0.0,0.0000000000000000,0.0,0.0
1,2025-01-02,0.2660654322172119,0.1434711695289264,0.1191659471041606,0.0597972432760881,0.1015446608462055,0.0464933320541111,0.0123125139914931,0.0503437909750871,0.0484441459592567,0.1500207873612843,0.0,0.0023409766861748,0.0,0.0
2,2025-01-03,0.2548218024162216,0.1225730137440547,0.0947137108947741,0.0716304299409555,0.1035778095629321,0.0467258266477275,0.0134965924444818,0.0501606030160702,0.0609103642149754,0.1813898471178069,0.0,0.0000000000000000,0.0,0.0
3,2025-01-04,0.2879887507937947,0.1097326801536181,0.1466932776920983,0.0664368441743022,0.1071925368169584,0.0483232029997883,0.0097372161238622,0.0429783180622335,0.0543258036227282,0.1238093078109407,0.0,0.0027820617496749,0.0,0.0
4,2025-01-05,0.0000000000000000,0.4447519693421333,0.0000000000000000,0.0000000000000000,0.2054502874175005,0.0349159037683627,0.0000000000000000,0.1466893761975729,0.0000000000000000,0.1681924632744305,0.0,0.0000000000000000,0.0,0.0


In [5]:
# check material validity
materials_14 = [str(i) for i in range(1, 15)]
materials_13 = [str(i) for i in range(1, 14)]


for c in [c for c in materials_14 if c in df.columns]:
    converted = pd.to_numeric(df[c], errors="coerce")
    newly_nan = converted.isna() & df[c].notna()

    if newly_nan.any():
        print(f"Column {c}: non-numeric values found")
        display(df.loc[newly_nan, [col for col in ["Datum", "Date", c] if col in df.columns]])

print(df[[c for c in materials_14 if c in df.columns]].dtypes)

1     float64
2     float64
3     float64
4     float64
5     float64
6     float64
7     float64
8     float64
9     float64
10    float64
11    float64
12    float64
13    float64
14    float64
dtype: object


In [6]:
audit = df.copy()

for c in [c for c in materials_14 if c in audit.columns]:
    audit[c] = pd.to_numeric(audit[c], errors="coerce")

date_col = "Datum" if "Datum" in audit.columns else "Date"

# Sum of the 13 official materials
audit["sum_1_13"] = audit[materials_13].sum(axis=1, min_count=13)
audit["diff_1_13"] = audit["sum_1_13"] - 1.0
audit["abs_diff_1_13"] = audit["diff_1_13"].abs()

# Sum including column 14, if present
if "14" in audit.columns:
    audit["sum_1_14"] = audit[materials_14].sum(axis=1, min_count=14)
    audit["diff_1_14"] = audit["sum_1_14"] - 1.0
    audit["residual_needed_for_13"] = 1.0 - audit["sum_1_13"]
    audit["col14_minus_residual"] = audit["14"] - audit["residual_needed_for_13"]

display(
    audit[
        [date_col] +
        materials_13 +
        (["14"] if "14" in audit.columns else []) +
        ["sum_1_13", "diff_1_13"] +
        (["sum_1_14", "diff_1_14", "residual_needed_for_13", "col14_minus_residual"]
         if "14" in audit.columns else [])
    ].head()
)

,Datum,1,2,3,4,5,6,7,8,9,10,11,12,13,14,sum_1_13,diff_1_13,sum_1_14,diff_1_14,residual_needed_for_13,col14_minus_residual
0,2025-01-01,0.2698455028712642,0.1270289971020676,0.1182996426476078,0.0616921791384429,0.1222465198143901,0.0430956317670275,0.0000000000000000,0.0464380322506089,0.0602521023343467,0.1511013920742439,0.0,0.0000000000000000,0.0,0.0,0.9999999999999997,-0.0000000000000003,0.9999999999999997,-0.0000000000000003,0.0000000000000003,-0.0000000000000003
1,2025-01-02,0.2660654322172119,0.1434711695289264,0.1191659471041606,0.0597972432760881,0.1015446608462055,0.0464933320541111,0.0123125139914931,0.0503437909750871,0.0484441459592567,0.1500207873612843,0.0,0.0023409766861748,0.0,0.0,0.9999999999999998,-0.0000000000000002,0.9999999999999998,-0.0000000000000002,0.0000000000000002,-0.0000000000000002
2,2025-01-03,0.2548218024162216,0.1225730137440547,0.0947137108947741,0.0716304299409555,0.1035778095629321,0.0467258266477275,0.0134965924444818,0.0501606030160702,0.0609103642149754,0.1813898471178069,0.0,0.0000000000000000,0.0,0.0,0.9999999999999998,-0.0000000000000002,0.9999999999999998,-0.0000000000000002,0.0000000000000002,-0.0000000000000002
3,2025-01-04,0.2879887507937947,0.1097326801536181,0.1466932776920983,0.0664368441743022,0.1071925368169584,0.0483232029997883,0.0097372161238622,0.0429783180622335,0.0543258036227282,0.1238093078109407,0.0,0.0027820617496749,0.0,0.0,0.9999999999999996,-0.0000000000000004,0.9999999999999996,-0.0000000000000004,0.0000000000000004,-0.0000000000000004
4,2025-01-05,0.0000000000000000,0.4447519693421333,0.0000000000000000,0.0000000000000000,0.2054502874175005,0.0349159037683627,0.0000000000000000,0.1466893761975729,0.0000000000000000,0.1681924632744305,0.0,0.0000000000000000,0.0,0.0,0.9999999999999999,-0.0000000000000001,0.9999999999999999,-0.0000000000000001,0.0000000000000001,-0.0000000000000001


In [8]:
TOL = 1e-10 # Same as code

bad_13 = audit["abs_diff_1_13"] > TOL

print("Rows where Materials 1-13 do not sum to 1:", bad_13.sum())
print("Maximum absolute deviation:", audit["abs_diff_1_13"].max())

cols_to_show = (
    [date_col] +
    materials_13 +
    (["14"] if "14" in audit.columns else []) +
    ["sum_1_13", "diff_1_13", "abs_diff_1_13"] +
    (["sum_1_14", "diff_1_14", "residual_needed_for_13", "col14_minus_residual"]
     if "14" in audit.columns else [])
)

display(audit.loc[bad_13, cols_to_show])

Rows where Materials 1-13 do not sum to 1: 1
Maximum absolute deviation: 0.001806903562975859


,Datum,1,2,3,4,5,6,7,8,9,10,11,12,13,14,sum_1_13,diff_1_13,abs_diff_1_13,sum_1_14,diff_1_14,residual_needed_for_13,col14_minus_residual
278,2025-10-24,0.2869601131002494,0.0960597588677267,0.0955395294902546,0.0631979413213032,0.1084062713452893,0.0360824742268041,0.0123862246437818,0.031313043063873,0.0519474846313916,0.1616404301621845,0.0438263466395564,0.0108334789446094,0.0,0.0018069035629755,0.9981930964370241,-0.0018069035629759,0.0018069035629759,0.9999999999999997,-0.0000000000000003,0.0018069035629759,-0.0000000000000004
